# Predictive Forecasting with Growth Analysis

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from config import db_config
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 7)

print("Libraries loaded.")

In [ ]:
engine = create_engine(db_config.url)
query = "SELECT * FROM nav_data"
df = pd.read_sql(query, engine)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df)} records and {df['scheme_code'].nunique()} unique funds from the database.")

## 2. Select a Fund to Forecast

In [ ]:
fund_list = sorted(df['scheme_name'].unique())
selected_fund_name = 'HDFC Flexi Cap Fund - Direct Plan - Growth Option'
if selected_fund_name not in fund_list:
    selected_fund_name = fund_list[0]

print(f"\nAnalyzing fund: {selected_fund_name}")

fund_df_raw = df[df['scheme_name'] == selected_fund_name][['date', 'nav']].copy()
fund_df_raw.drop_duplicates(subset=['date'], keep='last', inplace=True)

MIN_DATA_POINTS = 365 # We need at least a year of data for yearly analysis
has_enough_data = len(fund_df_raw) >= MIN_DATA_POINTS

if not has_enough_data:
    print(f"🛑 ANALYSIS HALTED: Not enough data for the selected fund. Need at least {MIN_DATA_POINTS} days of data for yearly analysis.")
else:
    print(f"✅ Sufficient data found. Proceeding with forecast...")
    fund_df = fund_df_raw.sort_values('date').set_index('date').asfreq('D').fillna(method='ffill').reset_index()

## 3. Forecasting Models & Detailed Visualization
This section will only run if the selected fund has enough data. It now includes the main forecast plot, a bar chart for historical growth, and a table with specific future NAV predictions.

In [ ]:
if has_enough_data:
    # --- Run Forecasting Models (Prophet & Holt-Winters) ---
    print("\nFitting forecasting models...")
    prophet_df = fund_df.rename(columns={'date': 'ds', 'nav': 'y'})
    model_prophet = Prophet(daily_seasonality=True).fit(prophet_df)
    future_prophet = model_prophet.make_future_dataframe(periods=365)
    forecast_prophet = model_prophet.predict(future_prophet)
    
    hw_df = fund_df.set_index('date')
    model_hw = ExponentialSmoothing(hw_df['nav'], trend='add', seasonal='add', seasonal_periods=365).fit()
    forecast_hw = model_hw.forecast(365)
    
    # --- Create Ensemble Forecast ---
    print("Creating ensemble forecast...")
    prophet_future_values = forecast_prophet[-365:]['yhat']
    hw_future_values = forecast_hw
    ensemble_forecast = (prophet_future_values.values + hw_future_values.values) / 2
    forecast_dates = pd.date_range(start=fund_df['date'].iloc[-1], periods=365 + 1)[1:]

    # --- NEW: Create a Summary Table of Forecasted Values ---
    print("\n--- Key Forecasted NAV Values ---")
    forecast_summary = pd.DataFrame({
        'Timeframe': ['1 Month', '3 Months', '6 Months', '1 Year'],
        'Date': [
            forecast_dates[29].date(),
            forecast_dates[89].date(),
            forecast_dates[179].date(),
            forecast_dates[364].date()
        ],
        'Forecasted NAV (₹)': [
            ensemble_forecast[29],
            ensemble_forecast[89],
            ensemble_forecast[179],
            ensemble_forecast[364]
        ]
    })
    forecast_summary['Forecasted NAV (₹)'] = forecast_summary['Forecasted NAV (₹)'].round(2)
    display(forecast_summary.set_index('Timeframe'))

    # --- Create Figure with Two Subplots ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 14), gridspec_kw={'height_ratios': [3, 2]})
    fig.suptitle(f'In-Depth Analysis for: {selected_fund_name}', fontsize=20)

    # --- Plot 1: The Main Forecast ---
    ax1.plot(fund_df['date'], fund_df['nav'], label='Historical NAV')
    ax1.plot(forecast_dates, ensemble_forecast, color='purple', linestyle='--', label='Ensemble Forecast (1-Year)')
    ax1.set_title('Historical NAV and Future Forecast')
    ax1.set_ylabel('NAV (₹)')
    ax1.legend()

    # --- Plot 2: Year-over-Year Growth Bar Chart ---
    print("\nCalculating year-over-year growth...")
    yearly_df = fund_df.set_index('date').resample('A-DEC').last()
    yearly_df['yearly_growth_pct'] = yearly_df['nav'].pct_change() * 100
    yearly_df.dropna(inplace=True)
    yearly_df.index = yearly_df.index.year
    
    colors = ['green' if x > 0 else 'red' for x in yearly_df['yearly_growth_pct']]
    sns.barplot(x=yearly_df.index, y=yearly_df['yearly_growth_pct'], ax=ax2, palette=colors)
    ax2.set_title('Historical Year-over-Year Growth')
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Growth (%)')
    for p in ax2.patches:
        ax2.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 9), textcoords='offset points')

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()